In [4]:
!pip install -U "transformers>=4.30.0" "accelerate>=0.20.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 66.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [1]:
!pip install transformers datasets sentencepiece accelerate pdfplumber nltk

In [1]:
# ==============================
# TRAINING BLOCK (RUN ONCE)
# ==============================
!pip install -q transformers datasets sentencepiece accelerate nltk torch

import torch
import nltk
from datasets import load_dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    TrainingArguments,
    Trainer
)

nltk.download("punkt", quiet=True)

# --------- Load SQuAD ---------
print("Loading SQuAD...")
dataset = load_dataset("squad")

model_name = "google/flan-t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

# --------- Preprocess: encourage more detailed questions ---------
def preprocess_qg(example):
    if not example["answers"]["text"]:
        return None
    answer = example["answers"]["text"][0]
    context = example["context"]

    # We add instruction text to push model towards more detailed questions
    input_text = (
        "Generate a detailed, clear question (around one full sentence) "
        "based on the following context and answer. "
        f"Context: {context} Answer: {answer}"
    )

    # Target is still the original SQuAD question (short),
    # but the model learns mapping from rich prompt → question
    target_text = example["question"]

    return {
        "input_text": input_text,
        "target_text": target_text,
    }

dataset = dataset.map(
    lambda x: preprocess_qg(x),
    num_proc=1,
    remove_columns=dataset["train"].column_names,
)
dataset = dataset.filter(lambda x: x is not None)

# --------- Tokenization with longer target length ---------
MAX_SOURCE_LEN = 512
MAX_TARGET_LEN = 96  # allow longer questions than SQuAD default

def tokenize_batch(batch):
    inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_SOURCE_LEN,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )
    targets = tokenizer(
        batch["target_text"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )
    labels = targets["input_ids"].clone()
    labels[labels == tokenizer.pad_token_id] = -100

    return {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],
        "labels": labels,
    }

tokenized_dataset = dataset.map(
    tokenize_batch,
    batched=True,
    batch_size=32,
    remove_columns=dataset["train"].column_names
)

small_train = tokenized_dataset["train"].shuffle(seed=42).select(range(20000))
small_val   = tokenized_dataset["validation"].shuffle(seed=42).select(range(5000))

# --------- Training ---------
device = "cuda" if torch.cuda.is_available() else "cpu"

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qg_results",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,  # increase later if Colab allows
    logging_dir="./logs",
    logging_steps=100,
    save_total_limit=2,
    fp16=(device == "cuda"),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
)

print("Starting training...")
trainer.train()

print("Saving trained model to 'qg_model_long'...")
model.save_pretrained("qg_model_long")
tokenizer.save_pretrained("qg_model_long")
print("Training complete.")

Loading SQuAD...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saving trained model to 'qg_model_long'...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete.


In [5]:
# ==============================
# TESTING BLOCK: LOGICAL Qs + LONG As FROM PASTED TEXT
# ==============================
!pip install -q transformers nltk torch

import torch
import nltk
import re
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Ensure NLTK data
for res in ["punkt_tab", "punkt"]:
    try:
        nltk.data.find(f"tokenizers/{res}/english.pickle")
    except LookupError:
        nltk.download(res, quiet=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

# --------- Load trained (or base) model ---------
# If you saved as "qg_model", change to that.
tokenizer = T5Tokenizer.from_pretrained("qg_model_long")
model = T5ForConditionalGeneration.from_pretrained("qg_model_long").to(device)

# ==============================
# PASTE TEXT HERE
# ==============================
context_text = """
Question–Answer (QA) generation is a key task in natural language processing that involves automatically creating relevant questions and their corresponding answers from a given text. It is widely used in applications such as chatbots, educational tools, and search engines to improve information retrieval and user interaction. Modern QA systems often leverage deep learning models, especially transformer-based architectures, to understand context and generate meaningful, coherent responses. By learning patterns in language and context, QA generation systems can produce accurate and human-like question–answer pairs, enhancing both learning experiences and automated support systems.
"""

print("Context sample:\n", context_text[:400])

# ==============================
# STEP 1: Extract candidate answer / topic phrases
# ==============================
def extract_candidate_answers(text: str, max_answers=5, min_len=2, max_len=8):
    try:
        sentences = nltk.sent_tokenize(text)
    except LookupError:
        sentences = text.split(".")

    answers = []

    for sent in sentences:
        sent = sent.strip()
        if len(sent.split()) < 6:
            continue

        clean = re.sub(r"[^0-9A-Za-z,\-\(\)\s]", " ", sent)
        tokens = clean.split()
        if len(tokens) < 6:
            continue

        # choose a 2–5 word phrase that looks like a concept or entity
        for i in range(len(tokens) - 1):
            phrase_tokens = tokens[i:i+4]
            phrase = " ".join(phrase_tokens)
            if any(ch.isalpha() for ch in phrase) and not phrase.strip().isdigit():
                if min_len <= len(phrase.split()) <= max_len:
                    answers.append(phrase.strip())
                    break

        if len(answers) >= max_answers:
            break

    if not answers and len(text.split()) > 5:
        answers.append(" ".join(text.split()[:5]))

    return answers

candidate_answers = extract_candidate_answers(context_text, max_answers=5)
print("\nCandidate topics/answers:", candidate_answers)

# ==============================
# STEP 2: Generate a short, logical question
# ==============================
def generate_logical_question(context: str, answer: str, max_length=64, num_beams=5):
    prompt = (
        "Write a clear, specific question based on the context and the given answer. "
        "The question should be short (one sentence) and logically point to the answer. "
        f"Context: {context} Answer: {answer}"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    outputs = model.generate(
        inputs.input_ids,
        max_length=max_length,
        num_beams=num_beams,
        temperature=1.0,
        early_stopping=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    question = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    return question

# ==============================
# STEP 3: Generate a long, detailed answer
# ==============================
def generate_long_answer(context: str, question: str, max_length=180, num_beams=5):
    prompt = (
        "Using the given context, write a detailed and well‑structured answer to the question. "
        "The answer should be around 50–60 words, covering the key idea clearly. "
        f"Context: {context} Question: {question}"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    outputs = model.generate(
        inputs.input_ids,
        max_length=max_length,
        num_beams=num_beams,
        temperature=1.0,
        early_stopping=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    return answer

# ==============================
# STEP 4: Build Q–A pairs
# ==============================
print("\n" + "="*60)
print("GENERATED LOGICAL QUESTIONS WITH LONG ANSWERS")
print("="*60 + "\n")

context_snippet = context_text[:900]

for topic in candidate_answers:
    # 1) Short logical question from topic + context
    question = generate_logical_question(context_snippet, topic)

    # 2) Long detailed answer to that question from the same context
    long_answer = generate_long_answer(context_snippet, question)

    print(f"Topic phrase: {topic}")
    print(f"Question: {question}")
    print(f"Answer (detailed): {long_answer}")
    print("-" * 60 + "\n")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Context sample:
 
Question–Answer (QA) generation is a key task in natural language processing that involves automatically creating relevant questions and their corresponding answers from a given text. It is widely used in applications such as chatbots, educational tools, and search engines to improve information retrieval and user interaction. Modern QA systems often leverage deep learning models, especially tran

Candidate topics/answers: ['Question Answer (QA) generation', 'It is widely used', 'Modern QA systems often', 'By learning patterns in']

GENERATED LOGICAL QUESTIONS WITH LONG ANSWERS

Topic phrase: Question Answer (QA) generation
Question: What is a key task in natural language processing that involves automatically creating relevant questions?
Answer (detailed): Question–Answer (QA) generation
------------------------------------------------------------

Topic phrase: It is widely used
Question: How is question–answer (Q&A) generated?
Answer (detailed): automatically creat